# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [1]:
import pandas as pd

import sqlite3


In [2]:
conexion = sqlite3.connect("medicare.db")

cant_tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",conexion)

cant_tablas

,name
0,pacientes
1,medicos
2,turnos


In [3]:
medicos = pd.read_sql("PRAGMA table_info(medicos)", conexion)
pacientes = pd.read_sql("PRAGMA table_info(pacientes)", conexion)
turnos = pd.read_sql("PRAGMA table_info(turnos)", conexion)


print(f"Información de cada tabla del DB: \n MÉDICOS: \n {medicos} \n \nPACIENTES: \n {pacientes} \n \nTURNOS: \n {turnos}")

Información de cada tabla del DB: 
 MÉDICOS: 
    cid          name     type  notnull dflt_value  pk
0    0            id  INTEGER        0       None   1
1    1        nombre     TEXT        0       None   0
2    2  especialidad     TEXT        0       None   0
3    3     matricula     TEXT        0       None   0
4    4        activo     TEXT        0       None   0 
 
PACIENTES: 
    cid              name     type  notnull dflt_value  pk
0    0                id  INTEGER        0       None   1
1    1            nombre     TEXT        0       None   0
2    2               dni     TEXT        0       None   0
3    3  fecha_nacimiento     TEXT        0       None   0
4    4            ciudad     TEXT        0       None   0
5    5       obra_social     TEXT        0       None   0 
 
TURNOS: 
    cid         name     type  notnull dflt_value  pk
0    0           id  INTEGER        0       None   1
1    1  paciente_id  INTEGER        0       None   0
2    2    medico_id  INTEGER       

In [4]:
df_pacientes =pd.read_sql("SELECT * FROM pacientes", conexion)
df_turnos = pd.read_sql("SELECT * FROM turnos", conexion)
df_medicos = pd.read_sql("SELECT * FROM medicos", conexion)

print(f" Cantidad de registros por tabla: \nPACIENTES: \n {df_pacientes.shape[0]} \n \nMEDICOS \n {df_medicos.shape[0]} \n \nTURNOS \n {df_turnos.shape[0]}" )


 Cantidad de registros por tabla: 
PACIENTES: 
 66 
 
MEDICOS 
 20 
 
TURNOS 
 210


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

In [5]:
# SOBRE LA DB:

#La base de datos contiene 3 tablas: Medicos, Turnos y Pacientes.


#Pacientes cuenta con 
#ID, 
#nombre, 
#DNI, 
#ciudad, 
#fecha_nacimiento y 
#su obra social.

#Medicos cuenta con 
#ID, 
#nombre, 
#especialidad, 
#matricula y 
#si está activo.

#Turnos cuenta con 
#ID, 
#paciente_id (clave foránea1), 
#medico_id (clave foránea2), 
#fecha, 
#hora, 
#motivo, 
#costo y 
#estado.

#Turnos es la tabla que está vinculada a las 2 tablas anteriores, reuniendo IDs de Médicos y Pacientes.

#Existen registros de 66 pacientes, 20 médicos y un total de 210 turnos.

#TIPOS DE DATOS: 

#Únicamente los ID son de tipo Int, las demás son STR.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [5]:
# tu codigo aquí

null_pac = df_pacientes.isnull().sum()
porcentaje_null_pac = (df_pacientes.isnull().sum() /len(df_pacientes) * 100).round(1)
dupl_pac = df_pacientes.duplicated(subset=["nombre","dni"]).sum()
dulp_pac_nombre = (df_pacientes[df_pacientes.duplicated(subset=["nombre", "dni"])])
type_data = df_pacientes.dtypes

print(f"Database pacientes \n \nCantidad de nulos x columna: \n {null_pac} \n \nPorcentaje de nulos x columna: \n {porcentaje_null_pac} \n \nDUPLICADOS: \n{dupl_pac} \n \nID Duplicados: \n {dulp_pac_nombre} \n\n Tipos de datos en la tabla: \n {type_data} ")


Database pacientes 
 
Cantidad de nulos x columna: 
 id                   0
nombre               0
dni                  7
fecha_nacimiento    11
ciudad               8
obra_social         12
dtype: int64 
 
Porcentaje de nulos x columna: 
 id                   0.0
nombre               0.0
dni                 10.6
fecha_nacimiento    16.7
ciudad              12.1
obra_social         18.2
dtype: float64 
 
DUPLICADOS: 
9 
 
ID Duplicados: 
     id               nombre         dni fecha_nacimiento        ciudad  \
3    4       Martina García  28.341.992       1985-04-12  Buenos Aires   
10  11         Roberto Díaz  31.882.441       1990-11-30       Córdoba   
13  14        Luciana Pérez  35.112.008       2001-07-22       Rosario   
45  46         Franco López  34.149.848              NaN           NaN   
51  52  Florencia Fernández  22.430.558       1956-06-19  Buenos Aires   
52  53        Pablo Álvarez  22.075.745       1975-01-08  Buenos Aires   
53  54      Carla Fernández  41.283.226

In [20]:
null_med = df_medicos.isnull().sum()
porcentaje_null_med = (df_medicos.isnull().sum() /len(df_medicos) * 100).round(1)
dupl_med = df_medicos.duplicated(subset=["nombre","matricula"]).sum()
dulp_med_nombre = (df_medicos[df_medicos.duplicated(subset=["nombre","matricula"])])
type_data_med = df_medicos.dtypes

print(f"Database médicos \n \nCantidad de nulos x columna: \n {null_med} \n \nPorcentaje de nulos x columna: \n {porcentaje_null_med} \n \nDUPLICADOS: \n{dupl_med} \n \nID Duplicados: \n {dulp_med_nombre} \n\n Tipos de datos en la tabla: \n {type_data_med} ")


Database médicos 
 
Cantidad de nulos x columna: 
 id              0
nombre          0
especialidad    0
matricula       2
activo          4
dtype: int64 
 
Porcentaje de nulos x columna: 
 id               0.0
nombre           0.0
especialidad     0.0
matricula       10.0
activo          20.0
dtype: float64 
 
DUPLICADOS: 
1 
 
ID Duplicados: 
    id             nombre    especialidad matricula activo
4   5  Dra. Beatriz Sosa  Clínica Médica  MP-12345      1 

 Tipos de datos en la tabla: 
 id              int64
nombre            str
especialidad      str
matricula         str
activo            str
dtype: object 


In [21]:
null_turn = df_turnos.isnull().sum()
porcentaje_null_turn = (df_turnos.isnull().sum() /len(df_turnos) * 100).round(1)
dupl_turn = df_turnos.duplicated(subset=["fecha","hora", "motivo", "paciente_id"]).sum()
dupl_turn_nombre = (df_turnos[df_turnos.duplicated(subset=["fecha","hora", "motivo", "paciente_id"])])
type_data_turn = df_turnos.dtypes

print(f"Database turnos \n \nCantidad de nulos x columna: \n {null_turn} \n \nPorcentaje de nulos x columna: \n {porcentaje_null_turn} \n \nDUPLICADOS: \n{dupl_turn} \n \nID Duplicados: \n {dupl_turn_nombre} \n\n Tipos de datos en la tabla: \n {type_data_turn} ")



Database turnos 
 
Cantidad de nulos x columna: 
 id              0
paciente_id     0
medico_id       0
fecha           0
hora           16
motivo         16
costo           9
estado          0
dtype: int64 
 
Porcentaje de nulos x columna: 
 id             0.0
paciente_id    0.0
medico_id      0.0
fecha          0.0
hora           7.6
motivo         7.6
costo          4.3
estado         0.0
dtype: float64 
 
DUPLICADOS: 
1 
 
ID Duplicados: 
     id  paciente_id  medico_id       fecha   hora         motivo costo  \
10  11            1          1  2024-03-01  09:00  Control anual  3500   

       estado  
10  realizado   

 Tipos de datos en la tabla: 
 id             int64
paciente_id    int64
medico_id      int64
fecha            str
hora             str
motivo           str
costo            str
estado           str
dtype: object 


In [22]:
#Prueba de errores en columna Costo en df_turnos.

pd.read_sql("SELECT * from turnos where costo < 0", conexion)


,id,paciente_id,medico_id,fecha,hora,motivo,costo,estado
0,18,5,6,2024-03-09,14:30,Dolor rodilla,-500,realizado
1,39,64,18,2024-03-29,13:45,Estudio de rutina,,realizado
2,59,39,1,2024-03-09,NaN,Chequeo general,-3500,pendiente
3,112,34,14,2024-05-09,14:15,Dolor de cabeza,-3500,pendiente
4,140,52,18,2024-05-13,NaN,Esguince tobillo,-500,realizado
5,174,38,5,2024-03-03,16:30,NaN,,realizado
6,181,18,1,2024-04-01,11:15,Fractura muñeca,-1000,realizado
7,191,60,19,2024-05-08,15:00,NaN,,realizado


**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

In [23]:
# Problemas en pacientes: 

# 7 nulos en dni, 
# 11 en fecha de nac, 
# 8 en ciudad, 
# 13 en obra social.
#  
# IDs 4, 
# 11, 
# 14, 
# 46, 
# 52, 
# 53, 
# 54, 
# 57 y 
# 64 duplicados.


# Problemas en médicos: 

# 2 nulos en matricula, 
# 4 nulos en activo. 
# 
# ID 5 duplicado.


# Problemas en Turnos: 
# 
# 16 nulos en hora, 
# 16 nulos en motivo, 
# 9 nulos en costo. 
# 
# ID 11 duplicado.
#
# Valores inválidos de costo: IDs 18, 39, 59, 112, 140, 174, 181, 191.


# El valor de "activo" en médicos podría ser un booleano en lugar de un STR, al haber unicamente dos opciones del valor.
# El valor "costo" en turnos debería ser int/float. Así en caso de realizar una operación como total pagado por x paciente, pueden ser operables.
# DNI en pacientes debería ser int en lugar de str, no por su operabilidad, sino por forzar a que el valor siempre sea numérico y no haya errores de datos.
# Existen valores negativos/vacíos en costo en registros de Turnos.

#

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 8 nulos | Cambiar valor a "Desconocida" | El valor no es correcto, pero no conocerlo no impediría el flow del sistema de turnos. |
| pacientes | dni | 7 nulo | Cambiar a "Desconocida". | El flow del sistema de turnos se mantiene, es posible corregirlo a futuro. |
| pacientes | obra social | 13 nulo | Cambiar a "Desconocida", crear código para mensaje de error. | Permite que el paciente solicite turno de igual forma. Así como permitiría hacer un código que en caso de introducir un paciente con un valor "Desconocida", salga un mensaje para correjirlo. |
| pacientes | ciudad de nacimiento | 11 nulo | Cambiar valor a "Desconocida" | El valor no es correcto, pero no conocerlo no impediría el flow del sistema de turnos. Posible de corregir a futuro. |
| pacientes | completa | 9 duplicados | Eliminar duplicados. | Los registros serían dobles en caso de dejarlos y solo estorban en la DB. |
| medicos | matricula | 2 nulo | Cambiar a "Desconocida", crear código para mensaje de error. | Permite sostener el registro y crear un mensaje de advertencia para corregir en el proximo ingreso al sistema. |
| medicos | activo | 4 nulo | Si es posible, en caso de tener algún turno agendado, que pase a activo. Sino dejar Inactivo hasta que se corrija el error. | En caso que un registro que tenia valor inválido sea activo, se pueden cruzar datos de asistencia de médico a la clinica. |
| medicos | completa | 1 duplicado | Eliminar duplicados. | Los registros serían dobles en caso de dejarlos y solo estorban en la DB. |
| medicos | activo | tipo incorrecto | Cambiar de STR a BOOL. | Unicamente hay 2 valores posibles que responden a si/no. |
| turnos | hora | 16 nulo | Cambiar el valor de nulo a "No hay datos" | Mantiene registro, es posible a futuro filtrarlos y corregir los datos. |
| turnos | motivo | 16 nulo | Cambiar el valor de nulo a "No hay datos" | Mantiene registro, es posible a futuro filtrarlos y corregir los datos.  |
| turnos | costo | 9 nulo | Cambiar el valor de nulo a "No hay datos" | Mantiene registro, es posible a futuro filtrarlos y corregir los datos.  |
| turnos | costos | 8 inválidos | En caso de numero negativo, eliminar el "-". | Lo más probable es que el valor fue introducido con un - que no iba. |
| turnos | completa | 1 duplicado | Eliminar duplicados. | Los registros serían dobles en caso de dejarlos y solo estorban en la DB. |
| turnos | costo | tipo incorrecto | cambiarlo a FLOAT | Permite su operabilidad. |



### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [41]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()
df_pacientes_limpio.drop_duplicates(subset=["nombre","dni"])
df_pacientes_limpio["ciudad"] = df_pacientes_limpio["ciudad"].fillna("Desconocida")
df_pacientes_limpio["dni"] = df_pacientes_limpio["dni"].fillna("Desconocida")
df_pacientes_limpio["obra_social"] = df_pacientes_limpio["obra_social"].fillna("Desconocida")
df_pacientes_limpio["fecha_nacimiento"] = df_pacientes_limpio["fecha_nacimiento"].fillna("Desconocida")

# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
df_medicos_limpio.drop_duplicates(subset=["nombre","matricula"])
df_medicos_limpio["matricula"] = df_medicos_limpio["matricula"].fillna("Desconocida")
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].fillna("Inactivo")

df_medicos_limpio["activo"] = df_medicos_limpio["activo"].astype("bool")
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].replace("Inactivo", False) #aca hice un replace porque me confundí yo al reemplazar los nulos con "Inactivo" en lugar de 0
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].map({True: "1", False: "0"})

# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
df_turnos_limpio.drop_duplicates(subset=["fecha","hora", "motivo", "paciente_id"])
df_turnos_limpio["hora"] = df_turnos_limpio["hora"].fillna("Desconocida")
df_turnos_limpio["motivo"] = df_turnos_limpio["motivo"].fillna("Desconocida")
df_turnos_limpio["costo"] = df_turnos_limpio["costo"].fillna("Desconocida")
df_turnos_limpio["costo"] = df_turnos_limpio["costo"].str.replace("-", "")




> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [37]:

print(df_turnos_limpio)

print(df_medicos_limpio)

      id  paciente_id  medico_id       fecha         hora              motivo  \
0      1            1          1  2024-03-01        09:00       Control anual   
1      2            2          2  2024-03-01        10:30              Fiebre   
2      3            3          1  2024-03-02        08:30     Dolor abdominal   
3      4            4          3  2024-03-02        11:00  Electrocardiograma   
4      5            5          2  2024-03-03  Desconocida  Control pediátrico   
..   ...          ...        ...         ...          ...                 ...   
205  206           47          3  2024-04-09        09:30       Dolor rodilla   
206  207           21         17  2024-04-07        11:15   Estudio de rutina   
207  208           46         19  2024-03-06        08:00    Esguince tobillo   
208  209           51          4  2024-05-15  Desconocida       Control anual   
209  210           35         18  2024-05-05        17:15   Estudio de rutina   

    costo      estado  
0  

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


In [ ]:
#Pregutna 1

pd.read_sql("SELECT medicos.nombre, medico_id, COUNT(*) as turnos FROM turnos inner join medicos on turnos.medico_id = medicos.id group by medico_id order by turnos DESC ", conexion)

,nombre,medico_id,turnos
0,Dra. Beatriz Sosa,1,20
1,Dr. Sebastián Ríos,6,18
2,Dra. Carmen Aguirre,10,14
3,Dra. Carmen Aguirre,3,14
4,Dra. Silvina Vega,20,13
5,Dr. Sebastián Ríos,12,13
6,Dra. Gabriela Luna,18,11
7,Dra. Andrea Blanco,14,11
8,Dr. Ezequiel Bravo,19,10
9,Dra. Laura Giménez,7,10


In [40]:
#Pregunta 2

especialidad_medica = pd.read_sql("SELECT id, nombre, especialidad from medicos group by especialidad", conexion)

facturado_por_especialidad = pd.read_sql("SELECT medicos.especialidad, sum(costo) from turnos inner join medicos on turnos.medico_id = medicos.id group by medicos.especialidad order by sum(costo) DESC", conexion)


print(facturado_por_especialidad)


     especialidad  sum(costo)
0   Traumatología    191300.0
1    Dermatología    181200.0
2  Clínica Médica    173300.0
3     Cardiología    154300.0
4       Pediatría    140100.0
5     Psiquiatría    104600.0
6     Ginecología     72800.0
7  Endocrinología     46800.0
8      Neurología     32700.0


In [14]:
#Pregunta 3

obra_social = pd.read_sql("Select obra_social, COUNT(*) as cantidad from pacientes group by obra_social order by cantidad DESC", conexion)

print(obra_social)

     obra_social  cantidad
0  Swiss Medical        14
1            NaN        12
2         Galeno        10
3           PAMI         9
4         Medife         8
5   Sancor Salud         6
6           OSDE         4
7           IOMA         3


In [43]:
#Pregunta 4

cancelados = pd.read_sql("SELECT count(*) from turnos where estado = 'cancelado'", conexion)

cancelados_medicos = pd.read_sql("SELECT medicos.nombre, medicos.id,count(*) from turnos inner join medicos on turnos.medico_id = medicos.id where estado = 'cancelado' group by medico_id order by count(*) DESC", conexion)

print(cancelados)
print(cancelados_medicos)

   count(*)
0        22
                 nombre  id  count(*)
0      Dr. Pablo Méndez   4         3
1      Dr. Hernán Costa  17         2
2    Dr. Sebastián Ríos   6         2
3     Dra. Beatriz Sosa   5         2
4     Dra. Beatriz Sosa   1         2
5    Dra. Gabriela Luna  18         1
6     Dra. Verónica Paz  16         1
7      Dr. Martín Cufré  15         1
8    Dra. Andrea Blanco  14         1
9    Dra. Laura Giménez  13         1
10   Dr. Sebastián Ríos  12         1
11  Dra. Carmen Aguirre  10         1
12     Dr. Nicolás Vera   9         1
13   Dra. Laura Giménez   7         1
14  Dra. Carmen Aguirre   3         1
15     Dr. Nicolás Vera   2         1


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá


#SEGUI TRABAJANDO CON DATOS DUPLICADOS Y ERRORES PORQUE NO MODIFIQUE LA CONEXION A LA NUEVA DB LIMPIA. La verdad que no se como tendria que linkear la nueva conexion, al ser .copy() de tablas de la db no entendi donde se guardaron.

REPORTE CLÍNICA MEDICARE


→ [Ir a la solución](./solucion.md)